# Audit dan pembagian PhiUSIIL

Penelitian ini menguji fitur leksikal dan pemisahan domain untuk deteksi phishing. Audit menggunakan URL sebagai teks lokal dan menampilkan hitungan agregat. Label sumber 0 berarti phishing dan dipetakan menjadi kelas positif 1.

[Data Kaggle](https://www.kaggle.com/datasets/kaggleprollc/phishing-url-websites-dataset-phiusiil) · [Dokumentasi UCI](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset) · [Artikel sumber](https://doi.org/10.1016/j.cose.2023.103545)

In [1]:
from hashlib import file_digest
from importlib.metadata import version
from ipaddress import ip_address
from itertools import combinations
from pathlib import Path
from urllib.parse import urlsplit
import json
import platform

import numpy as np
import pandas as pd
import tldextract
from sklearn.model_selection import GroupShuffleSplit, train_test_split

SEED = 42
SOURCE = Path("Dataset/PhiUSIIL_Phishing_URL_Dataset.csv")
OUT = Path("hasil")
OUT.mkdir(exist_ok=True)
PARTS = ("latih", "validasi", "uji")
GROUPS = ("akar", "path", "query")
PAGE = ["LineOfCode", "LargestLineLength", "HasTitle", "HasFavicon", "Robots",
        "IsResponsive", "NoOfURLRedirect", "NoOfSelfRedirect", "HasDescription",
        "NoOfPopup", "NoOfiFrame", "HasExternalFormSubmit", "HasSocialNet",
        "HasSubmitButton", "HasHiddenFields", "HasPasswordField", "HasCopyrightInfo",
        "NoOfImage", "NoOfCSS", "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef", "NoOfExternalRef"]
LEXICAL = ["panjang_url", "panjang_host", "panjang_path", "panjang_query", "titik_host",
           "huruf_url", "angka_url", "karakter_lain", "strip_host", "segmen_path",
           "persen_url", "sama_dengan_query", "ampersand_query", "https", "host_ip"]
with SOURCE.open("rb") as file:
    sha256 = file_digest(file, "sha256").hexdigest()
raw = pd.read_csv(SOURCE, usecols=["URL", "label", *PAGE], dtype={"URL": "string"})
if raw["URL"].isna().any() or not raw["label"].isin([0, 1]).all():
    raise ValueError("Periksa URL kosong dan label di luar 0/1.")
raw["phishing"] = (raw["label"] == 0).astype("int8")
conflicts = raw.groupby("URL")["phishing"].nunique()
bad_urls = conflicts.index[conflicts > 1]
clean = raw.loc[~raw["URL"].isin(bad_urls)].drop_duplicates("URL").copy()
audit = dict(baris_mentah=len(raw), url_unik=len(conflicts), duplikat_tambahan=len(raw)-len(conflicts),
             url_konflik_label=len(bad_urls), baris_bersih=len(clean))
display(pd.Series(audit, name="jumlah").to_frame())

,jumlah
baris_mentah,235795
url_unik,235370
duplikat_tambahan,425
url_konflik_label,0
baris_bersih,235370


## Fitur dan kelompok struktur

A memakai fitur leksikal dan atribut halaman numerik yang tersedia di CSV. B dan C memakai fitur leksikal di bawah ini. Daftar fitur mengecualikan pengenal, teks halaman, skor kemiripan dan probabilitas yang asal statistiknya tidak dipastikan. Query tidak kosong mendapat prioritas, kemudian path dengan segmen, lalu akar.

Domain terdaftar memakai snapshot Public Suffix List bawaan tldextract dengan suffix privat disertakan. Alamat IP dan host dengan suffix tak dikenali memakai host utuh.

In [2]:
extract_domain = tldextract.TLDExtract(cache_dir=None, suffix_list_urls=(),
                                      include_psl_private_domains=True)

def describe_url(url):
    parsed = urlsplit(url)
    if not parsed.hostname:
        raise ValueError("Ada URL tanpa hostname.")
    host = parsed.hostname.rstrip(".").encode("idna").decode("ascii")
    try:
        ip_address(host)
        is_ip, domain = 1, host
    except ValueError:
        is_ip = 0
        domain = extract_domain(host).top_domain_under_public_suffix or host
    path, query = parsed.path, parsed.query
    segments = [segment for segment in path.split("/") if segment]
    group = "query" if query else "path" if segments else "akar"
    letters = sum(char.isalpha() for char in url)
    digits = sum(char.isdigit() for char in url)
    return (domain, group, len(url), len(host), len(path), len(query), host.count("."),
            letters, digits, len(url)-letters-digits, host.count("-"), len(segments),
            url.count("%"), query.count("="), query.count("&"), int(parsed.scheme == "https"), is_ip)

features = pd.DataFrame.from_records(
    (describe_url(url) for url in clean["URL"]),
    columns=["domain_terdaftar", "struktur", *LEXICAL], index=clean.index,
)
groups_by_url = pd.Series(features["struktur"].to_numpy(), index=clean["URL"])
raw_groups = raw["URL"].map(groups_by_url)
missing = raw_groups.isna()
if missing.any():
    raw_groups.loc[missing] = raw.loc[missing, "URL"].map(lambda url: describe_url(url)[1])
counts = pd.crosstab(raw_groups, raw["phishing"]).reindex(index=GROUPS, columns=[0, 1], fill_value=0)
counts = counts.rename(columns={0: "sah", 1: "phishing"}).rename_axis("struktur")
counts.to_csv(OUT / "audit_struktur_mentah.csv")
display(counts)
data = pd.concat([clean[["phishing", *PAGE]], features], axis=1).reset_index(drop=True)

phishing,sah,phishing
struktur,,
akar,134850,72460
path,0,22406
query,0,6079


## Split dengan target 70/15/15

A dan B berbagi split acak terstratifikasi. C memakai GroupShuffleSplit dua tahap. Rasio C diterapkan pada jumlah domain sehingga rasio baris bisa berbeda. Audit mencatat distribusi dan irisan setiap pasangan bagian.

In [3]:
ids = np.arange(len(data))
y = data["phishing"].to_numpy()
domains = data["domain_terdaftar"].to_numpy()
train, hold = train_test_split(ids, test_size=.30, random_state=SEED, stratify=y)
val, test = train_test_split(hold, test_size=.50, random_state=SEED, stratify=y[hold])
gtrain, ghold = next(GroupShuffleSplit(test_size=.30, random_state=SEED).split(ids, groups=domains))
gval, gtest = next(GroupShuffleSplit(test_size=.50, random_state=SEED).split(ghold, groups=domains[ghold]))

splits = {"acak": (train, val, test), "domain": (gtrain, ghold[gval], ghold[gtest])}
summary, overlap_rows, distribution = [], [], []
for split, parts in splits.items():
    assert np.array_equal(np.sort(np.concatenate(parts)), ids)
    domain_sets = {}
    for part, rows in zip(PARTS, parts):
        assert set(y[rows]) == {0, 1}
        data.loc[rows, split] = part
        domain_sets[part] = set(domains[rows])
        summary.append(dict(split=split, bagian=part, baris=len(rows), proporsi=len(rows)/len(data),
                            sah=int((y[rows] == 0).sum()), phishing=int(y[rows].sum()),
                            domain_unik=len(domain_sets[part])))
        counts = data.loc[rows].groupby(["struktur", "phishing"]).size()
        for group in GROUPS:
            distribution.append(dict(split=split, bagian=part, struktur=group,
                sah=int(counts.get((group, 0), 0)), phishing=int(counts.get((group, 1), 0))))
    for first, second in combinations(PARTS, 2):
        shared = domain_sets[first] & domain_sets[second]
        if split == "domain":
            assert not shared
        second_domains = data.loc[data[split] == second, "domain_terdaftar"]
        overlap_rows.append(dict(split=split, bagian_1=first, bagian_2=second, domain_irisan=len(shared),
                                 proporsi_baris_bagian_2=second_domains.isin(shared).mean()))

for name, rows in {"ringkasan_split": summary, "irisan_domain": overlap_rows,
                   "distribusi_struktur_split": distribution}.items():
    table = pd.DataFrame(rows)
    table.to_csv(OUT / f"{name}.csv", index=False)
    display(table)

data.drop(columns="domain_terdaftar").to_csv(OUT / "fitur_internal.csv.gz", index=False)
manifest = dict(sha256=sha256, seed=SEED, audit=audit, fitur_leksikal=LEXICAL, fitur_halaman=PAGE,
                python=platform.python_version(),
                paket={pkg: version(pkg) for pkg in ("numpy", "pandas", "scikit-learn", "tldextract")})
(OUT / "manifest_data.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

,split,bagian,baris,proporsi,sah,phishing,domain_unik
0,acak,latih,164759,0.700000,94395,70364,140191
1,acak,validasi,35305,0.149998,20227,15078,31418
2,acak,uji,35306,0.150002,20228,15078,31495
3,domain,latih,164785,0.700110,94076,70709,138396
4,domain,validasi,34356,0.145966,20289,14067,29656
5,domain,uji,36229,0.153924,20485,15744,29657


,split,bagian_1,bagian_2,domain_irisan,proporsi_baris_bagian_2
0,acak,latih,validasi,2576,0.179550
1,acak,latih,uji,2581,0.177817
2,acak,validasi,uji,1184,0.131054
3,domain,latih,validasi,0,0.000000
4,domain,latih,uji,0,0.000000
5,domain,validasi,uji,0,0.000000


,split,bagian,struktur,sah,phishing
0,acak,latih,akar,94395,50577
1,acak,latih,path,0,15537
2,acak,latih,query,0,4250
3,acak,validasi,akar,20227,10793
4,acak,validasi,path,0,3374
5,acak,validasi,query,0,911
6,acak,uji,akar,20228,10842
7,acak,uji,path,0,3326
8,acak,uji,query,0,910
9,domain,latih,akar,94076,51047


1192